# skills mirror updater

Converts the opencode `SKILL.md` files into the static espanso trigger
mirror at `match/skills.yml`. Run all cells top-to-bottom after adding or
removing skills (the daily cron does this automatically).

In [1]:
import importlib
import tempfile
from pathlib import Path

import yaml
import skills_lib
importlib.reload(skills_lib)

skills = skills_lib.collect_skills()
print(f"Found {len(skills)} skills under {skills_lib.DEFAULT_SKILLS_DIR}")
print("output target:", skills_lib.DEFAULT_OUTPUT)
skills[:3]

Found 38 skills under /Users/jensbay/.config/opencode/skills
output target: /Users/jensbay/Library/Application Support/espanso/match/skills.yml


[{'id': 'brainstorming',
  'name': 'brainstorming',
  'description': 'You MUST use this before any creative work - creating features, building components, adding functionality, modifying behavior, or planning anything new, in software or out of it (a talk, a business, a renovation).',
  'path': '/Users/jensbay/.config/opencode/skills/brainstorming/SKILL.md'},
 {'id': 'claude-code',
  'name': 'claude-code',
  'description': 'Run Claude Code from the terminal and track consumption/usage',
  'path': '/Users/jensbay/.config/opencode/skills/claude-code/SKILL.md'},
 {'id': 'diagnosing-superpowers',
  'name': 'diagnosing-superpowers',
  'description': 'Use when a superpowers session went wrong and your human partner wants to know why — repeated work, ignored plans, stumbles, poor results, a skill that didn\'t fire, "it took too long", "why is it so expensive", "what is it doing" — or wants to build a bug report for the superpowers maintainers, for the current session or a past one identified 

In [2]:
# Sanity checks: unique names, every skill has a description
assert len(skills) > 0, "no skills found"
missing = [s["id"] for s in skills if not s["name"] or not s["description"]]
assert not missing, f"skills missing name/description: {missing}"
names = [s["name"] for s in skills]
assert len(names) == len(set(names)), "duplicate skill names would collide"
print("All skills unique and have name + description" )

All skills unique and have name + description


In [3]:
# Generate to a temp file and validate it round-trips as YAML
with tempfile.TemporaryDirectory() as d:
    tmp = Path(d) / "skills.yml"
    skills_lib.generate_skills_yml(skills, output_path=tmp)
    parsed = yaml.safe_load(tmp.read_text())
    triggers = {m["trigger"]: m["replace"] for m in parsed["matches"]}
assert ":skills" in triggers
assert ":tdd" in triggers and triggers[":tdd"].startswith("Test-driven development")
assert len(parsed["matches"]) == len(skills) + 1  # + the :skills listing
print("Generated mirror valid:", len(parsed["matches"]), "matches")

Generated mirror valid: 39 matches


In [4]:
# Preview the :skills expansion
print(skills_lib.format_skill_list(skills))

opencode skills (38):
  :ask-matt -- Ask which skill or flow fits your situation
  :brainstorming -- You MUST use this before any creative work - creating features, building components, adding functionality, modifying behavior, or planning anything new, in software or out of it (a talk, a business, a renovation)
  :claude-code -- Run Claude Code from the terminal and track consumption/usage
  :code-review -- Review the changes since a fixed point (commit, branch, tag, or merge-base) along two axes: Standards (does the code follow this repo's documented coding standards?) and Spec (does the code match what the originating issue/spec asked for?)
  :codebase-design -- Shared vocabulary for designing deep modules
  :diagnosing-bugs -- Diagnosis loop for hard bugs and performance regressions
  :diagnosing-superpowers -- Use when a superpowers session went wrong and your human partner wants to know why — repeated work, ignored plans, stumbles, poor results, a skill that didn't fire, "it took

In [5]:
# Regenerate the real mirror at match/skills.yml
out = skills_lib.generate_skills_yml(skills)
print(f"Wrote {out}")
print(out.read_text()[:600])

Wrote /Users/jensbay/Library/Application Support/espanso/match/skills.yml
# AUTO-GENERATED by skills_lib.py -- do not edit by hand.
# Mirror of ~/.config/opencode/skills (md -> yml). Regenerate daily via sync-daily.sh
# or manually: python3 skills_lib.py sync
# version: 1.1.0 | skills at generation time (38): ask-matt, brainstorming, claude-code, code-review, codebase-design, diagnosing-bugs, diagnosing-superpowers, dispatching-parallel-agents, domain-modeling, executing-plans, finishing-a-development-branch, grill-with-docs, grilling, implement, implement-spec, improve-codebase-architecture, pr, prototype, receiving-code-review, requesting-code-review, research, re


In [6]:
# End-to-end: run the same CLI command the cron runs
import subprocess, sys

r = subprocess.run([sys.executable, "skills_lib.py", "sync"], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print(r.stdout)
print("CLI e2e OK")

Wrote /Users/jensbay/Library/Application Support/espanso/match/skills.yml (38 skills, version 1.1.0)

CLI e2e OK
